# Graph-of-Thought — Colab GPU runner

Fine-tunes a small sentiment classifier (DistilBERT on 100 SST-2 samples) on a **T4**,
following `AGENTS.md` §31. Set **Runtime → Change runtime type → T4 GPU** first.

Data/model come from the Hugging Face Hub (public — no token needed). The run writes
`output/<run_id>/` (manifest, telemetry, dashboard, and **model weights** under
`checkpoints/final`); the download cell zips it so you can copy it back into the repo.

In [ ]:
# 1) Clone the repo
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git

%cd graph-of-thought

In [ ]:
# 2) Install training deps (Colab already ships a CUDA build of torch)
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) Fine-tune (live loss / learning-rate graphs + log print in the cell)
!python scripts/train.py --config configs/train_sst2.json

In [ ]:
# 4) Zip the latest run (incl. model weights) and download it
import glob, os
from google.colab import files
latest = sorted(glob.glob('output/*/'))[-1].rstrip('/')
run_id = os.path.basename(latest)
!zip -qr {run_id}.zip {latest}
print('Downloading', run_id + '.zip')
files.download(f'{run_id}.zip')

In [ ]:
# 5) Show the HTML dashboard inline
from IPython.display import HTML
HTML(open(os.path.join(latest, 'dashboard.html')).read())

### CPU demo (no GPU needed)
`!python scripts/run.py --config configs/example_run.json`

---
## Reasoning-graph POC — Llama 3.2 1B on GSM8K (T4 GPU)

Runs the graph-generation POC (`sbc_t4_poc_research_plan.md`): samples token-level
reasoning chains for GSM8K questions with a **frozen** `meta-llama/Llama-3.2-1B-Instruct`
(FP16), records per-token hidden states + log-probabilities, consolidates them into a
DAG under a sweep of latent-merge heuristics/thresholds, and renders a standalone HTML
graph report. Dataset and model are chosen entirely in `configs/graph_gsm8k.json`.

`Llama-3.2` is a **gated** model — store a Hugging Face token as a Colab secret named
`HF_TOKEN` (and request access on the model page) before running the next cell.


In [ ]:
# A) Authenticate for the gated Llama model, then generate reasoning graphs on GSM8K
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))  # HF_TOKEN Colab secret; never committed (AGENTS.md section 8)

!python scripts/generate_graphs.py --config configs/graph_gsm8k.json


In [ ]:
# B) Zip the graph run and preview the sampled graph report inline
import glob, os, json
from google.colab import files
from IPython.display import HTML

latest = sorted(glob.glob('output/*graph-gsm8k*/'))[-1].rstrip('/')
run_id = os.path.basename(latest)
!zip -qr {run_id}.zip {latest}
files.download(f'{run_id}.zip')

manifest = json.load(open(os.path.join(latest, 'run_manifest.json')))
report = manifest['outputs']['reports'][0]
print('Graph report:', report)
HTML(open(os.path.join(latest, report)).read())
